## CART (Classification And Regression Tree)

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import (mean_absolute_error, mean_squared_error, 
    mean_absolute_percentage_error, r2_score, accuracy_score, precision_score, 
    recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix)
from sklearn.base import BaseEstimator

### Реализация

In [31]:
class Tree_Node:
    def __init__(self, feature=None, threshold=None, left_tree=None, right_tree=None, rating_tree=None):
        self.feature = feature     
        self.threshold = threshold 
        self.left_tree = left_tree       
        self.right_tree = right_tree         
        self.rating_tree = rating_tree

In [29]:
class Custom_Tree(BaseEstimator):
    def __init__(self, mode):
        self.max_depth = 5
        self.min_samples_split = 10
        self.min_samples_leaf = 10
        self.mode = mode
        self.root = None

    def mse(self, y):
        mse_num = np.mean(np.power(y - np.mean(y), 2))
        return mse_num

    def gini(self, y):
        frequencies = np.unique(y, return_counts=True)[1] 
        probabilities = frequencies / len(y)
        return 1 - np.sum(np.power(probabilities, 2))

    def criterion(self, y):
        if self.mode == 'regression':
            return self.mse(y=y)
        else: 
            return self.gini(y=y)
        
    def split(self, x, y):
        index = None
        threshold = None
        best_gain = -np.inf

        n, num_of_features = x.shape

        if n < self.min_samples_split:
            return None, None  
        
        parent_loss = self.criterion(y=y)
        
        for i in range(num_of_features):
            column_i = x[:, i]
            treshholds_i = np.unique(column_i)

            for t in treshholds_i:
                left_indices = np.where(column_i <= t)[0]
                right_indices = np.where(column_i > t)[0]

                if len(left_indices) < self.min_samples_leaf or len(right_indices) < self.min_samples_leaf:
                    continue

                loss_left_tree = self.criterion(y=y[left_indices])
                loss_right_tree = self.criterion(y=y[right_indices])

                gain = parent_loss - (len(left_indices) / n) * loss_left_tree - (len(right_indices) / n) * loss_right_tree
                if gain > best_gain:
                    best_gain = gain
                    index = i
                    threshold = t

        return index, threshold
    
    def build_tree(self, x, y, depth=0):
        n, num_of_features = x.shape
        num_of_lables = len(np.unique(y))

        if (depth >= self.max_depth or num_of_lables == 1 or n < self.min_samples_split):
            leaf_value = self.calculate_leaf_value(y)
            return Tree_Node(rating_tree=leaf_value)
        
        index, threshold = self.split(x=x, y=y)

        if index is None:
            return Tree_Node(rating_tree=self.calculate_leaf_value(y))
        
        left_tree = np.where(x[:, index] <= threshold)[0]
        right_tree = np.where(x[:, index] > threshold)[0]

        left_child = self.build_tree(x=x[left_tree, :], y=y[left_tree], depth=depth + 1)    
        right_child = self.build_tree(x=x[right_tree, :], y=y[right_tree], depth=depth + 1)

        return Tree_Node(feature=index, threshold=threshold, left_tree=left_child, right_tree=right_child, rating_tree=None)
    
    def calculate_leaf_value(self, y):
        if self.mode == 'regression':
            return np.mean(y)
        else:
            counts = np.bincount(y.astype(int))
            return np.argmax(counts)
        
    def fit(self, x, y):
        x = np.array(x)
        y = np.array(y)
        self.root = self.build_tree(x, y)
        return self 

    def predict_one(self, x, node):
        if node.rating_tree is not None:
            return node.rating_tree
        
        if x[node.feature] <= node.threshold:
            return self.predict_one(x, node.left_tree)
        else:
            return self.predict_one(x, node.right_tree)

    def predict(self, X):
        X = np.array(X)
        return np.array([self.predict_one(x, self.root) for x in X])

### Задача регрессии

In [4]:
regression = pd.read_csv('../data/diamonds_filtered.csv')
y_regression = regression['price']
x_regression = regression.drop('price', axis=1)

In [32]:
model_reg = Custom_Tree(mode='regression')
kf = KFold(n_splits=5, shuffle=True, random_state=81)

y_pred = cross_val_predict(model_reg, x_regression, y_regression, cv=kf, n_jobs=-1)

metrics_reg = {
    'Model': 'Regression Tree',
    'MAE': round(mean_absolute_error(y_regression, y_pred), 4),
    'RMSE': round(np.sqrt(mean_squared_error(y_regression, y_pred)), 4),
    'MAPE': round(mean_absolute_percentage_error(y_regression, y_pred), 4),
    'R2': round(r2_score(y_regression, y_pred), 5)
}

results_reg = pd.DataFrame([metrics_reg])
results_reg

,Model,MAE,RMSE,MAPE,R2
0,Regression Tree,513.9607,898.4966,0.1532,0.9373


### Задача классификации

In [38]:
classification = pd.read_csv(f"../data/credit_card_fraud_filtered.csv")
y_classification = classification['fraud'][:10_000]
x_classification = classification.drop('fraud', axis=1)[:10_000]

In [39]:
model_clf = Custom_Tree(mode='classification')

kf = KFold(n_splits=5, shuffle=True, random_state=81)

y_pred = cross_val_predict(model_clf, x_classification, y_classification, cv=kf, n_jobs=-1)

metrics_clf = {
    "Model": 'Classification Tree',
    "Accuracy": accuracy_score(y_classification, y_pred),
    "Precision": precision_score(y_classification, y_pred),
    "Recall": recall_score(y_classification, y_pred),
    "F1-Score": f1_score(y_classification, y_pred),
    "ROC-AUC": roc_auc_score(y_classification, y_pred)
}

print(classification_report(y_classification, y_pred))
print(confusion_matrix(y_classification, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9366
           1       0.99      0.99      0.99       634

    accuracy                           1.00     10000
   macro avg       1.00      0.99      0.99     10000
weighted avg       1.00      1.00      1.00     10000

[[9361    5]
 [   8  626]]


In [40]:
results_clf = pd.DataFrame([metrics_clf])
results_clf

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Classification Tree,0.9987,0.992076,0.987382,0.989723,0.993424


### Вывод

- Несмотря на сопоставимую точность, кастомная реализация уступает библиотечной в скорости обучения на больших выборках (например, в задаче классификации).
- Использование ограничений `max_depth` и `min_samples_leaf` позволило избежать переобучения, что подтверждается стабильностью метрик на кросс-валидации.